# Number of visits per year for 5.3.6

- update : 2026-09-09
- kernel : conda_py313_opsim53
- Ce notebook : download from : https://github.com/lsst-pst/survey_strategy/blob/main/fbs_5.3/v5.3_Update.ipynb and other files
- summary.h5 : https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- Opsim baseline runs : https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/baseline/
- Opsim other runs : https://s3df.slac.stanford.edu/data/rubin/sim-data/
- Table of simulations : https://usdf-maf.slac.stanford.edu/
- LSST survey strategy : https://github.com/lsst-pst/survey_strategy/

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp
import skyproj
import rubin_sim.maf as maf

In [ ]:
path_topsim = os.getenv("RUBIN_SIM_DATA_DIR")
path_summary = os.path.join(path_topsim, "maf/fbs5.3.6/summary.h5")
print(path_topsim, " -- ", path_summary)

In [ ]:
# opsdb = "baseline_v5.3.6_10yrs.db"
opsdb = os.path.join(path_topsim, "sim_baseline/baseline_v5.3.6_10yrs.db")
run_name = opsdb.replace(".db", "")
out_dir = run_name + "_nvis"

In [ ]:
metric = maf.CountMetric(col="observationStartMJD", metric_name="Nvisits")
slicer = maf.HealpixSlicer(nside=64)

bundle_dict = {}
for year in range(1, 11):
    filterlist, colors, orders, sqls, info_labels = maf.filter_list(
        all=True, extra_sql=f"night < 365*{year}", extra_info_label=f"year {year}", band_col="band"
    )
    for band in filterlist:
        bundle = maf.MetricBundle(
            metric,
            slicer,
            sqls[band],
            info_label=info_labels[band],
            run_name=run_name,
            summary_metrics=maf.extended_summary(),
            plot_dict={"color": colors[band], "percentile_clip": 98},
        )
        bundle_dict[f"y{year} {band}"] = bundle

    filterlist, colors, orders, sqls, info_labels = maf.filter_list(
        all=True,
        extra_sql=f"night < 365*{year} and scheduler_note not like '%DD%' and scheduler_note not like '%ToO%'",
        extra_info_label=f"year {year} no too no ddf",
        band_col="band",
    )
    for band in filterlist:
        bundle = maf.MetricBundle(
            metric,
            slicer,
            sqls[band],
            info_label=info_labels[band],
            run_name=run_name,
            summary_metrics=maf.extended_summary(),
            plot_dict={"color": colors[band], "percentile_clip": 98},
        )
        bundle_dict[f"y{year} {band} no_too no_ddf"] = bundle

In [ ]:
results_db = maf.ResultsDb(out_dir=out_dir)
g = maf.MetricBundleGroup(bundle_dict, opsdb, out_dir=out_dir, results_db=results_db)

In [ ]:
g.run_all()

### Summary values

In [ ]:
from IPython.display import HTML


def side_by_side(*dfs):
    html = '<div style="display:flex">'
    for df in dfs:
        html += '<div style="margin-right: 2em">'
        html += df.to_html()
        html += "</div>"
    html += "</div>"
    display(HTML(html))


summary_type = "Median"

summaries = []
for year in range(1, 11):
    for band in filterlist:
        val = bundle_dict[f"y{year} {band}"].summary_values[summary_type]
        summaries.append([year, band, val])
summaries = pd.DataFrame(summaries, columns=["year", "band", "val"])
summaries = summaries.pivot(index=["year"], columns=["band"], values=["val"]).droplevel(0, axis=1)
summaries = summaries[list(filterlist)].rename_axis(columns={"band": "all visits"})

summaries_n = []
for year in range(1, 11):
    for band in filterlist:
        val = bundle_dict[f"y{year} {band} no_too no_ddf"].summary_values[summary_type]
        summaries_n.append([year, band, val])
summaries_n = pd.DataFrame(summaries_n, columns=["year", "band", "val"])
summaries_n = summaries_n.pivot(index=["year"], columns=["band"], values=["val"]).droplevel(0, axis=1)
summaries_n = summaries_n[list(filterlist)].rename_axis(columns={"band": "no too/ddf"})

side_by_side(summaries, summaries_n)

### Plot all visits

In [ ]:
ph = maf.PlotHandler(out_dir=out_dir, thumbnail=False, fig_format="png")

In [ ]:
for year in range(1, 11):
    for band in filterlist:
        ph.set_metric_bundles([bundle_dict[f"y{year} {band}"]])
        ph.plot(
            plot_func=maf.HpxmapPlotter(),
            plot_dicts={
                "figsize": (8, 5),
                "xlabel": None,
                "fontsize": "x-large",
                "skyproj": skyproj.McBrydeSkyproj,
            },
        )

In [ ]:
for year in range(1, 11):
    for band in filterlist:
        ph.set_metric_bundles([bundle_dict[f"y{year} {band} no_too no_ddf"]])
        ph.plot(
            plot_func=maf.HpxmapPlotter(),
            plot_dicts={
                "figsize": (8, 5),
                "xlabel": None,
                "fontsize": "x-large",
                "skyproj": skyproj.McBrydeSkyproj,
            },
        )